# Putting many TinyRNNs onto one GPU

Here we investigate a potential speed-up obtained by training many TinyRNNs with the same training data. This means we can run all hyperparameters on the same loop. Could be an upwards of 100x speed up.

In [1]:
%load_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [154]:
from NM_TinyRNN.code.models import training_fast as speedrun
from NM_TinyRNN.code.models import datasets as ds
from NM_TinyRNN.code.models import rnns
# write some code to further parallelise the training and test it here
from NM_TinyRNN.code.models import nested_cv as nc
from NM_TinyRNN.code.models import nested_cv_io as save_data
from NM_TinyRNN.code.models import nested_jobs

import numpy as np
import pandas as pd
import torch #for testing a few things
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path
from importlib import reload


CODE_DIR = Path('.') ## OBS THIS MAY NEED TO BE ADJUSTED!
SAVE_PATH = CODE_DIR/'NM_TinyRNN/data/rnns'
DATA_PATH = Path('./NM_TinyRNN/data/AB_behaviour/')

%load_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [172]:

# let's test some code!
test_data_path = DATA_PATH / "WS16"
test_save_path  = './NM_TinyRNN/data/rnns/gpu_test'
reload(save_data)
reload(ds)
reload(nc)
reload(speedrun)
reload(rnns)
trainer = speedrun.TrainerGPU(weight_seeds = list(range(1,11)),
                        sparsity_lambdas = [1,0.1,1e-2,1e-3,1e-4,1e-5,1e-6],
                        energy_lambdas = [1,0.1,1e-2,1e-3],
                        hebbian_lambdas = [0.0])
model = rnns.TinyRNN(rnn_type = 'vanilla', hidden_size = 2)
dataset = ds.AB_Dataset(test_data_path, sequence_length = 64)

#final_state_dict, config = trainer.fit(model, dataset)


Sequence length 64 excludes 7.1% of trials


In [184]:
splits = nc.nested_cv_splits(dataset)
trials_df = save_data.get_model_trial_by_trial_df(model, dataset,splits['inner_folds'][0])
outer_results = nc.run_outer_fold(model, dataset,
                                  outer_loop_number = 1,
                                  n_outer_loops = 10,
                                  save_path = test_save_path, 
                                  trainer_kwargs = {'sparsity_lambdas':[1e-1,1e-2,1e-3,1e-4,1e-5],
                                                    'energy_lambdas':[0.1,1e-2,1e-3]})
print([d['val_loss'] for d in outer_results['inner_results']])


[outer 1/10]  outer eval: 11 blocks  |  9 inner folds  |  saving to NM_TinyRNN/data/rnns/gpu_test
Parallelizing 150 models on cpu
Parallelizing 150 models on cpu
Parallelizing 150 models on cpu
Parallelizing 150 models on cpu
Parallelizing 150 models on cpu
Parallelizing 150 models on cpu
Parallelizing 150 models on cpu
Parallelizing 150 models on cpu


 14%|█▍        | 142/1000 [01:05<06:32,  2.19it/s]

Search complete. Best model index: 117. Val. loss: 0.5111628174781799
  Saved outer_fold_1/inner_fold_7 -> NM_TinyRNN/data/rnns/gpu_test/outer_fold_1/inner_fold_7  (val_loss=0.5112, eval_loss=0.6344)
Parallelizing 150 models on cpu


 28%|██▊       | 281/1000 [02:05<05:20,  2.24it/s]


[0.47655779123306274, 0.5073539018630981, 0.48726028203964233, 0.4491475820541382, 0.48639070987701416, 0.4798084795475006, 0.4782405197620392, 0.5111628174781799, 0.4766395092010498]


In [186]:
test_df = nested_jobs.run_training(overwrite=True, test= False)

Submitting model training for WS20 to HPC
Submitted batch job 2766116
Submitting model training for WS20 to HPC
Submitted batch job 2766117
Submitting model training for WS20 to HPC
Submitted batch job 2766118
Submitting model training for WS20 to HPC
Submitted batch job 2766119
Submitting model training for WS20 to HPC
Submitted batch job 2766120
Submitting model training for WS20 to HPC
Submitted batch job 2766121
Submitting model training for WS20 to HPC
Submitted batch job 2766122
Submitting model training for WS20 to HPC
Submitted batch job 2766123
Submitting model training for WS20 to HPC
Submitted batch job 2766124
Submitting model training for WS20 to HPC
Submitted batch job 2766125
Submitting model training for WS20 to HPC
Submitted batch job 2766126
Submitting model training for WS20 to HPC
Submitted batch job 2766127
Submitting model training for WS20 to HPC
Submitted batch job 2766128
Submitting model training for WS20 to HPC
Submitted batch job 2766129
Submitting model tra